In [1]:
import pyspark
from pyspark.sql.types import IntegerType,BooleanType,DateType
from pyspark.sql.functions import year
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
from pyspark.sql import functions as F
from pyspark.sql.functions import split
from pyspark.sql.functions import substring

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv')

In [4]:
#df['FiscalYear'] = [2009 if x == 'male' else 'U' for x in df['Sex']]
#df['AllocationDate'] = [201404 if x == 'male' else 202203 for x in df['Sex']]
#df.to_csv('/Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv')

In [5]:
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FiscalYear,AllocationDate
0,0,0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2009,201404
1,1,1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,U,202203
2,2,2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,U,202203
3,3,3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,U,202203
4,4,4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,2009,201404


In [6]:
from pyspark.sql import SparkSession

In [7]:
spark = SparkSession.builder.appName('Practices').getOrCreate()

22/04/21 09:28:48 WARN Utils: Your hostname, Abhimanyus-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
22/04/21 09:28:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/04/21 09:28:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
spark

In [9]:
df1 = spark.read.csv('/Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv')

In [10]:
df2 = spark.read.option('header', 'true').csv('/Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv', inferSchema=True)

In [11]:
df2.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Unnamed: 0.1: integer (nullable = true)
 |-- Unnamed: 0: integer (nullable = true)
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)
 |-- FiscalYear: string (nullable = true)
 |-- AllocationDate: integer (nullable = true)



In [12]:
df2.columns

['_c0',
 'Unnamed: 0.1',
 'Unnamed: 0',
 'PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked',
 'FiscalYear',
 'AllocationDate']

In [13]:
df2.head(3)

22/04/21 09:28:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
 Schema: _c0, Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
Expected: _c0 but found: 
CSV file: file:///Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv


[Row(_c0=0, Unnamed: 0.1=0, Unnamed: 0=0, PassengerId=1, Survived=0, Pclass=3, Name='Braund, Mr. Owen Harris', Sex='male', Age=22.0, SibSp=1, Parch=0, Ticket='A/5 21171', Fare=7.25, Cabin=None, Embarked='S', FiscalYear='2009', AllocationDate=201404),
 Row(_c0=1, Unnamed: 0.1=1, Unnamed: 0=1, PassengerId=2, Survived=1, Pclass=1, Name='Cumings, Mrs. John Bradley (Florence Briggs Thayer)', Sex='female', Age=38.0, SibSp=1, Parch=0, Ticket='PC 17599', Fare=71.2833, Cabin='C85', Embarked='C', FiscalYear='U', AllocationDate=202203),
 Row(_c0=2, Unnamed: 0.1=2, Unnamed: 0=2, PassengerId=3, Survived=1, Pclass=3, Name='Heikkinen, Miss. Laina', Sex='female', Age=26.0, SibSp=0, Parch=0, Ticket='STON/O2. 3101282', Fare=7.925, Cabin=None, Embarked='S', FiscalYear='U', AllocationDate=202203)]

In [14]:
df2.show(3)

+---+------------+----------+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+--------------+
|_c0|Unnamed: 0.1|Unnamed: 0|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|FiscalYear|AllocationDate|
+---+------------+----------+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+--------------+
|  0|           0|         0|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| null|       S|      2009|        201404|
|  1|           1|         1|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|         U|        202203|
|  2|           2|         2|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| null|       S|         U|  

22/04/21 09:28:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
 Schema: _c0, Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
Expected: _c0 but found: 
CSV file: file:///Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv


In [15]:
type(df2.select('Name'))

pyspark.sql.dataframe.DataFrame

In [16]:
df2.select('FiscalYear')

DataFrame[FiscalYear: string]

In [17]:
x = df2.select(['Sex', 'Age'])

In [18]:
x.dtypes

[('Sex', 'string'), ('Age', 'double')]

In [19]:
#df2.filter('Pclass >= 3').show()

In [20]:
df2.dtypes

[('_c0', 'int'),
 ('Unnamed: 0.1', 'int'),
 ('Unnamed: 0', 'int'),
 ('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'double'),
 ('Cabin', 'string'),
 ('Embarked', 'string'),
 ('FiscalYear', 'string'),
 ('AllocationDate', 'int')]

In [21]:
df2 = df2.filter('Fare >= 11.1333')

In [22]:
final = df2.limit(20)

In [23]:
df2.dtypes

[('_c0', 'int'),
 ('Unnamed: 0.1', 'int'),
 ('Unnamed: 0', 'int'),
 ('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'double'),
 ('Cabin', 'string'),
 ('Embarked', 'string'),
 ('FiscalYear', 'string'),
 ('AllocationDate', 'int')]

In [24]:
df2 = df2.withColumn('Fare', df2.Fare.cast(IntegerType()))

In [25]:
df2.dtypes

[('_c0', 'int'),
 ('Unnamed: 0.1', 'int'),
 ('Unnamed: 0', 'int'),
 ('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'int'),
 ('Cabin', 'string'),
 ('Embarked', 'string'),
 ('FiscalYear', 'string'),
 ('AllocationDate', 'int')]

In [26]:
df2.dtypes

[('_c0', 'int'),
 ('Unnamed: 0.1', 'int'),
 ('Unnamed: 0', 'int'),
 ('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'int'),
 ('Cabin', 'string'),
 ('Embarked', 'string'),
 ('FiscalYear', 'string'),
 ('AllocationDate', 'int')]

In [27]:
df.dtypes

Unnamed: 0.2        int64
Unnamed: 0.1        int64
Unnamed: 0          int64
PassengerId         int64
Survived            int64
Pclass              int64
Name               object
Sex                object
Age               float64
SibSp               int64
Parch               int64
Ticket             object
Fare              float64
Cabin              object
Embarked           object
FiscalYear         object
AllocationDate      int64
dtype: object

In [28]:
#df2 = df2.filter(df2['AllocationDate'] == 2014)
#df2.limit(100).show()

In [29]:
#df2 = df2.withColumn('AllocationDate', df2.AllocationDate.cast(DateType()))
#df2 = df2.withColumn('AllocationDate',year(df2['AllocationDate']))
#df2.show()

In [30]:
df2 = df2.withColumn('AllocationDate', df2.AllocationDate.cast("string"))
df2.dtypes

[('_c0', 'int'),
 ('Unnamed: 0.1', 'int'),
 ('Unnamed: 0', 'int'),
 ('PassengerId', 'int'),
 ('Survived', 'int'),
 ('Pclass', 'int'),
 ('Name', 'string'),
 ('Sex', 'string'),
 ('Age', 'double'),
 ('SibSp', 'int'),
 ('Parch', 'int'),
 ('Ticket', 'string'),
 ('Fare', 'int'),
 ('Cabin', 'string'),
 ('Embarked', 'string'),
 ('FiscalYear', 'string'),
 ('AllocationDate', 'string')]

In [34]:
df2 = df2.withColumn("AllocationDate", F.to_date(F.col("AllocationDate").cast("string"),'yyyyMM'))

In [35]:
df2 = df2.withColumn('AllocationDate',year(df2['AllocationDate']))
df2.show()

+---+------------+----------+-----------+--------+------+--------------------+------+----+-----+-----+--------+----+-----------+--------+----------+--------------+
|_c0|Unnamed: 0.1|Unnamed: 0|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|Fare|      Cabin|Embarked|FiscalYear|AllocationDate|
+---+------------+----------+-----------+--------+------+--------------------+------+----+-----+-----+--------+----+-----------+--------+----------+--------------+
|  1|           1|         1|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599|  71|        C85|       C|         U|          2022|
|  3|           3|         3|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|  53|       C123|       S|         U|          2022|
|  6|           6|         6|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|   17463|  51|        E46|       S|      2009|          2014|
|  7|           

22/04/21 09:29:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
 Schema: _c0, Unnamed: 0.1, Unnamed: 0, PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, FiscalYear, AllocationDate
Expected: _c0 but found: 
CSV file: file:///Users/abhimanyu_yadav/Desktop/Practice/Titanic/Titanic.csv


In [33]:
#df2 = df2.filter('AllocationDate <= 2022')
#df2.show()